In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/ingestion_autoloader/ingestion_demo/raw/customers/"
    )
)


In [0]:
source_path = "/Volumes/ingestion_autoloader/ingestion_demo/raw/input"
schema_path = "/Volumes/ingestion_autoloader/ingestion_demo/raw/schema"

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)
)

display(
    df,
    checkpointLocation="/Volumes/ingestion_autoloader/ingestion_demo/raw/checkpoint/test"
)


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/ingestion_autoloader/ingestion_demo/raw/schema"
    )
)


In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = (
    df
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", df["_metadata"]["file_path"])
)
display(
    bronze_df,
    checkpointLocation="/Volumes/ingestion_autoloader/ingestion_demo/raw/checkpoint/display_test"
)


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ingestion_autoloader.bronze;

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = (
    df
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", df["_metadata"]["file_path"])
    .withColumnRenamed("Customer ID", "customer_id")
)


In [0]:
%sql
SELECT *
FROM ingestion_autoloader.bronze.customers;


In [0]:
bronze_checkpoint = "/Volumes/ingestion_autoloader/ingestion_demo/raw/checkpoint/bronze"

query = (
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", bronze_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("ingestion_autoloader.bronze.customers")
)


In [0]:
silver_df = (
    spark.readStream
    .table("ingestion_autoloader.bronze.customers")
)
print(silver_df.isStreaming)



In [0]:
from pyspark.sql.functions import col, lower, trim

silver_df = (
    silver_df
    .filter(col("customer_id").isNotNull())
    .withColumn("name", trim(col("name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("city", trim(col("city")))
)
display(
    silver_df,
    checkpointLocation="/Volumes/ingestion_autoloader/ingestion_demo/raw/checkpoint/silver_display"
)


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ingestion_autoloader.silver")

silver_checkpoint = (
    "/Volumes/ingestion_autoloader/"
    "ingestion_demo/raw/checkpoint/silver"
)
silver_query = (
    silver_df.writeStream
    .format("delta")
    .option("checkpointLocation", silver_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("ingestion_autoloader.silver.customers")
)


In [0]:
%sql
SELECT *
FROM ingestion_autoloader.silver.customers;
